# 예제 franka_ex12: FR3 웨이포인트 순차 추적

여러 개의 끝단 자세 (waypoints) 를 순서대로 방문하는 예제.
6-DOF 용 `ex12_waypoint_follow.py` 의 YAML 로딩/단계별 진행 흐름을
주피터 노트북 형태로 옮기되, **노트북 셀이 곧 각 단계의 트리거**가 되게 한다.

**6-DOF 예제와 다른 점**
- 웨이포인트를 FR3 워크스페이스에 맞춰 X≈0.45m, Y±0.20m, Z 0.40~0.60m 로 키움
- 끝단 링크 `fr3_hand_tcp`, planning group `fr3_arm`
- `home` 없음 → 시작/복귀는 `ready`
- 7-DOF redundancy 덕에 IK 시드 기반 연속 이동이 6-DOF 보다 더 자연스럽다

**학습 내용**
- `compute_ik` 서비스로 IK 시드 (현재 조인트 → 다음 자세) → joint goal 변환 → 최소 이동
- 같은 trajectory 를 FK 로 끝단 좌표 시퀀스로 변환해 RViz 에 누적 표시
- 단계별 마커 색상 (대기/진행/성공/실패/지난)

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/waypoint_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/waypoint_markers'

## 2. 초기화

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetPositionIK, GetPositionFK
from moveit_msgs.msg import RobotState
from visualization_msgs.msg import Marker, MarkerArray
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Point, Vector3

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex12_waypoint_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex12 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)
ik_client = node.create_client(GetPositionIK, 'compute_ik')

## 3. 서버 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup / Execute / FK 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

In [ ]:
from moveit_msgs.action import ExecuteTrajectory

execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
if not execute_client.wait_for_server(timeout_sec=15.0):
    raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')

def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

In [ ]:
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import RobotState

fk_client = node.create_client(GetPositionFK, 'compute_fk')
fk_client.wait_for_service(timeout_sec=10.0)

def trajectory_to_ee_path(trajectory, max_points: int = 60):
    '''RobotTrajectory → fr3_hand_tcp 의 base 기준 좌표 리스트.'''
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts

## 6. IK 시드 헬퍼

현재 조인트 상태를 시드로 IK 를 풀어, 다음 자세도 가까운 IK 해를 우선 채택하게 한다.
이러면 7-DOF 로봇이 같은 자세를 향할 때 손목이 휙 회전하는 일이 줄어든다.

In [ ]:
def get_current_joints() -> dict:
    if joint_state['msg'] is None:
        return {}
    msg = joint_state['msg']
    return {n: p for n, p in zip(msg.name, msg.position) if n in ARM_JOINTS}

def solve_seeded_ik(pose: Pose, timeout_sec: float = 0.5):
    if not ik_client.wait_for_service(timeout_sec=2.0):
        return None
    req = GetPositionIK.Request()
    req.ik_request.group_name = PLANNING_GROUP
    req.ik_request.pose_stamped.header.frame_id = REFERENCE_FRAME
    req.ik_request.pose_stamped.pose = pose
    req.ik_request.avoid_collisions = True
    req.ik_request.timeout.sec = 0
    req.ik_request.timeout.nanosec = int(timeout_sec * 1e9)
    cur = get_current_joints()
    if cur:
        rs = RobotState()
        rs.joint_state.name = list(cur.keys())
        rs.joint_state.position = list(cur.values())
        req.ik_request.robot_state = rs
    fut = ik_client.call_async(req)
    rclpy.spin_until_future_complete(node, fut, timeout_sec=2.0)
    res = fut.result()
    if res is None or res.error_code.val != MoveItErrorCodes.SUCCESS:
        return None
    sol = {n: p for n, p in zip(res.solution.joint_state.name, res.solution.joint_state.position)
           if n in ARM_JOINTS}
    return sol

def plan_via_ik(pose: Pose):
    seed = solve_seeded_ik(pose)
    if seed is not None:
        return plan_to_joint_goal(seed)
    node.get_logger().warn('IK 실패 → plan_to_pose_goal 폴백')
    return plan_to_pose_goal(pose)

## 7. 마커 헬퍼

waypoints 를 sphere + 라벨 + 연결선으로 발행. 상태 변화 시 색상 바꿈.

In [ ]:
COLOR_PENDING = ColorRGBA(r=1.0, g=1.0, b=0.0, a=0.85)
COLOR_ACTIVE  = ColorRGBA(r=0.2, g=0.5, b=1.0, a=0.95)
COLOR_SUCCESS = ColorRGBA(r=0.0, g=1.0, b=0.0, a=0.85)
COLOR_FAIL    = ColorRGBA(r=1.0, g=0.0, b=0.0, a=0.85)
COLOR_PASSED  = ColorRGBA(r=0.4, g=0.4, b=0.4, a=0.6)
COLOR_LINK    = ColorRGBA(r=0.5, g=0.5, b=1.0, a=0.5)
COLOR_TEXT    = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
COLOR_EE      = ColorRGBA(r=1.0, g=0.5, b=0.0, a=0.95)

waypoints = []   # [{'name', 'pose', 'pause_sec'}]
wp_status = {}   # idx -> 'pending'|'active'|'success'|'fail'|'passed'

def color_for(status):
    return {
        'pending': COLOR_PENDING, 'active': COLOR_ACTIVE,
        'success': COLOR_SUCCESS, 'fail': COLOR_FAIL,
        'passed': COLOR_PASSED,
    }.get(status, COLOR_PENDING)

def publish_waypoint_markers():
    ma = MarkerArray()
    stamp = node.get_clock().now().to_msg()
    for i, wp in enumerate(waypoints):
        s = wp_status.get(i, 'pending')
        c = color_for(s)
        scale = 0.05 if s == 'active' else 0.035
        sphere = Marker()
        sphere.header.frame_id = REFERENCE_FRAME
        sphere.header.stamp = stamp
        sphere.ns = 'waypoints'
        sphere.id = i
        sphere.type = Marker.SPHERE
        sphere.action = Marker.ADD
        sphere.pose = wp['pose']
        sphere.scale = Vector3(x=scale, y=scale, z=scale)
        sphere.color = c
        text = Marker()
        text.header.frame_id = REFERENCE_FRAME
        text.header.stamp = stamp
        text.ns = 'waypoint_labels'
        text.id = i
        text.type = Marker.TEXT_VIEW_FACING
        text.action = Marker.ADD
        text.pose.position = Point(x=wp['pose'].position.x,
                                   y=wp['pose'].position.y,
                                   z=wp['pose'].position.z + 0.07)
        text.pose.orientation.w = 1.0
        text.scale.z = 0.04
        text.color = COLOR_TEXT
        text.text = f"{i+1}"
        ma.markers.extend([sphere, text])
    if len(waypoints) >= 2:
        line = Marker()
        line.header.frame_id = REFERENCE_FRAME
        line.header.stamp = stamp
        line.ns = 'waypoint_link'
        line.id = 0
        line.type = Marker.LINE_STRIP
        line.action = Marker.ADD
        line.pose.orientation.w = 1.0
        line.scale.x = 0.004
        line.color = COLOR_LINK
        line.points = [wp['pose'].position for wp in waypoints]
        ma.markers.append(line)
    marker_pub.publish(ma)

def publish_ee_path(ee_points, ns='ee_path_line', color=COLOR_EE):
    if not ee_points:
        return
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = ns
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.007
    line.color = color
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]
    ma = MarkerArray(markers=[line])
    marker_pub.publish(ma)

def set_status(idx: int, s: str):
    wp_status[idx] = s
    publish_waypoint_markers()

## 8. 웨이포인트 정의 (FR3 스케일)

원본 YAML 의 6-DOF 좌표(`x≈0.20m, y±0.06m, z≈0.30~0.40m`)를 FR3 워크스페이스로 환산:
`x≈0.45m, y±0.20m, z 0.40~0.60m`. 모두 그리퍼 아래(`roll=π`).

In [ ]:
waypoints = [
    {'name': 'P1 Start',     'pose': make_pose(0.45,  0.00, 0.55, math.pi, 0.0, 0.0), 'pause_sec': 1.0},
    {'name': 'P2 Right',     'pose': make_pose(0.50, -0.20, 0.45, math.pi, 0.0, 0.0), 'pause_sec': 0.5},
    {'name': 'P3 Low Right', 'pose': make_pose(0.50, -0.15, 0.40, math.pi, 0.0, 0.0), 'pause_sec': 0.5},
    {'name': 'P4 Low Left',  'pose': make_pose(0.50,  0.15, 0.40, math.pi, 0.0, 0.0), 'pause_sec': 0.5},
    {'name': 'P5 Left',      'pose': make_pose(0.50,  0.20, 0.45, math.pi, 0.0, 0.0), 'pause_sec': 0.5},
    {'name': 'P6 Top',       'pose': make_pose(0.40,  0.00, 0.65, math.pi, 0.0, 0.0), 'pause_sec': 1.0},
]
for i in range(len(waypoints)):
    wp_status[i] = 'pending'
publish_waypoint_markers()
node.get_logger().info(f'총 {len(waypoints)} 웨이포인트 등록')

## 9. 초기화 — `ready` 자세로 이동

In [ ]:
import time
ok, traj = plan_to_joint_goal(ready_target)
if ok:
    pts = trajectory_to_ee_path(traj)
    publish_ee_path(pts)
    execute_trajectory(traj)
time.sleep(1.0)

## 10. 1부 — 개별 웨이포인트 순차 이동 (셀마다 한 점씩)

이 섹션에서는 **셀 한 번 = 다음 웨이포인트 1개 이동**.
내부적으로 IK 시드 → joint goal 로 plan + execute. RViz 에서 끝단 경로가 주황색 LINE_STRIP 으로 표시된다.

In [ ]:
def go_to_wp(idx: int) -> bool:
    wp = waypoints[idx]
    set_status(idx, 'active')
    node.get_logger().info(
        f"[{idx+1}/{len(waypoints)}] {wp['name']} "
        f"({wp['pose'].position.x:.2f}, {wp['pose'].position.y:.2f}, {wp['pose'].position.z:.2f})"
    )
    ok, traj = plan_via_ik(wp['pose'])
    if not ok or traj is None:
        set_status(idx, 'fail')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    exec_ok = execute_trajectory(traj)
    set_status(idx, 'success' if exec_ok else 'fail')
    if wp['pause_sec'] > 0:
        time.sleep(wp['pause_sec'])
    if exec_ok and idx < len(waypoints) - 1:
        set_status(idx, 'passed')
    return exec_ok

### 10-1. P1

In [ ]:
go_to_wp(0)

### 10-2. P2

In [ ]:
go_to_wp(1)

### 10-3. P3

In [ ]:
go_to_wp(2)

### 10-4. P4

In [ ]:
go_to_wp(3)

### 10-5. P5

In [ ]:
go_to_wp(4)

### 10-6. P6

In [ ]:
go_to_wp(5)

## 11. 2부 — 전체 자동 일주 (시안 LINE_STRIP 누적)

상태를 모두 `pending` 으로 리셋한 뒤 모든 점을 자동으로 일주.
각 segment 의 trajectory 를 FK 로 변환해 시안색 LINE_STRIP 에 누적한다 → 전체 끝단 경로가 한눈에 보인다.

In [ ]:
COLOR_SMOOTH_PATH = ColorRGBA(r=0.0, g=1.0, b=1.0, a=0.95)

# 상태 리셋
for i in range(len(waypoints)):
    wp_status[i] = 'pending'
publish_waypoint_markers()
time.sleep(0.5)

# ready 로 한 번 더 정렬
ok, traj = plan_to_joint_goal(ready_target)
if ok:
    execute_trajectory(traj)
time.sleep(0.5)

running_pts = []
for i, wp in enumerate(waypoints):
    set_status(i, 'active')
    node.get_logger().info(f"  [{i+1}/{len(waypoints)}] {wp['name']}")
    ok, traj = plan_via_ik(wp['pose'])
    if ok and traj is not None:
        seg = trajectory_to_ee_path(traj)
        running_pts.extend(seg)
        publish_ee_path(running_pts, ns='smooth_segment_path', color=COLOR_SMOOTH_PATH)
        seg_ok = execute_trajectory(traj)
        set_status(i, 'success' if seg_ok else 'fail')
    else:
        set_status(i, 'fail')
    time.sleep(0.5)

node.get_logger().info(f'2부 일주 완료 — 누적 끝단 경로 {len(running_pts)} 점')

## 12. ready 복귀

In [ ]:
ok, traj = plan_to_joint_goal(ready_target)
if ok:
    execute_trajectory(traj)
node.get_logger().info('=== franka_ex12 완료! ===')

## 13. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass